In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy.stats import multivariate_normal
from tqdm import tqdm
import cvxpy as cp
from Newsvendor import Newsvendor
import torch
import torch.nn as nn
import normflows as nf
from torch.utils.data import TensorDataset, DataLoader, random_split
from pathlib import Path
from time import perf_counter

In [ ]:
########################## Normalizing flow function ###################################
def train_nf_model(
    latent_size, best_K, hidden_node, hidden_layer, num_bins, block_size,
    total_epoch, x, device, base_gmm, batch_size=64, lr=1e-3,
    patience=30, val_split=0.2
):

    means = base_gmm.means_.astype(np.float32)
    weights = base_gmm.weights_.astype(np.float32)

    if base_gmm.covariance_type == "diag":
        scale = np.sqrt(base_gmm.covariances_).astype(np.float32)
    elif base_gmm.covariance_type == "full":
        diag_cov = np.array([np.diag(C) for C in base_gmm.covariances_])
        scale = np.sqrt(diag_cov).astype(np.float32)
    else:
        raise ValueError(f"Unsupported covariance_type: {base_gmm.covariance_type}")

    flows = [nf.flows.AutoregressiveRationalQuadraticSpline(latent_size, hidden_layer, hidden_node, num_bins=num_bins) for _ in range(block_size)]
    q0 = nf.distributions.GaussianMixture(n_modes=best_K, dim=latent_size, loc=means, scale=scale, weights=weights, trainable=False)
    nfm = nf.NormalizingFlow(q0=q0, flows=flows).to(device)
    optimizer = torch.optim.Adam(nfm.parameters(), lr=lr)

    dataset = TensorDataset(x)
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    loss_hist = []
    val_loss_hist = []

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    best_epoch = 0

    for epoch in tqdm(range(total_epoch), desc="Training NF", leave=False):
        nfm.train()
        train_loss_epoch = 0.0
        for batch in train_loader:
            x_batch = batch[0].to(device)
            optimizer.zero_grad()
            loss = nfm.forward_kld(x_batch)
            if not torch.isnan(loss):
                loss.backward()
                optimizer.step()
                train_loss_epoch += loss.item()

        nfm.eval()
        val_loss_epoch = 0.0
        with torch.no_grad():
            for batch in val_loader:
                x_batch = batch[0].to(device)
                loss = nfm.forward_kld(x_batch)
                if not torch.isnan(loss):
                    val_loss_epoch += loss.item()

        loss_hist.append(train_loss_epoch)
        val_loss_hist.append(val_loss_epoch)

        if val_loss_epoch < best_val_loss:
            best_val_loss = val_loss_epoch
            best_epoch = epoch + 1
            patience_counter = 0
            best_model_state = {k: v.detach().cpu().clone() for k, v in nfm.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_model_state is not None:
        nfm.load_state_dict(best_model_state)

    stopped_epoch = len(loss_hist)
    training_info = {
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "early_stopped": stopped_epoch < total_epoch,
        "best_val_loss": best_val_loss,
    }

    return nfm, loss_hist, training_info

def inverse(nfm, x):
    with torch.no_grad():
        z_np = nfm.inverse(x).detach().cpu().numpy()
    return z_np

def forward(nfm, z):
    with torch.no_grad():
        x = nfm.forward(z).detach().cpu().numpy()
    return x

In [ ]:
######################### GMM function ###################################
def make_eps_grid(a_list, b_list):
    eps_grid = []

    for b in b_list:
        for a in a_list:
            eps_grid.append(a * (10 ** b))

    return eps_grid

def NewsVendor_2_Wass(xi, eps, h, b):
    scale = 100 
    xi = xi.astype(float)/ scale
    eps = eps / scale
    N = xi.shape[0]
    lda = cp.Variable(nonneg = True)
    s = cp.Variable(N)
    theta = cp.Variable(nonneg = True)
    q = cp.Variable(nonneg = True)

    const = []
    for i in range(N):
        const.append(cp.norm2(cp.hstack([2 * lda * xi[i] + theta - h, lda * (xi[i]**2) - h * q + s[i] - lda])) <= lda * (xi[i]**2) - h * q + s[i] +lda)
        const.append(cp.norm2(cp.hstack([2 * lda * xi[i] + theta + b, lda * (xi[i]**2) + b * q + s[i] - lda])) <= lda * (xi[i]**2) + b * q + s[i] +lda)
        const.append(lda * (xi[i]**2) - h * q + s[i] >= 0)
        const.append(lda * (xi[i]**2) + b * q + s[i] >= 0)

    obj = cp.Minimize(lda * (eps**2) + (1 / N) * cp.sum(s))
    prob = cp.Problem(obj, const)
    prob.solve(solver = cp.MOSEK, verbose = False)

    return q.value * scale 

def generate_data(n, dim_s, dim_xi, W1, W2, seed=None):
    rng = np.random.default_rng(seed)  
    s = rng.uniform(-2, 2, size=(n, dim_s))
    eps1 = rng.uniform(-2, 2, size=(n, dim_xi))
    eps2 = rng.uniform(-2, 2, size=(n, dim_xi))
    lin  = s @ W1.T
    quad = (s**2) @ W2.T
    xi1 = lin + 50 + eps1
    xi2 = quad + 40 + eps2
    score = s[:, 0]
    prob = 0.5 * (1 + np.tanh(score)) 
    u = rng.random(n)
    xi = np.where(u[:, None] < prob[:, None], xi1, xi2)
    return s, xi

def transforming_conditional(s, num_components, mu_k, sig_k, p_k, dim_s):
    reg = 1e-6
    eig_floor = 1e-10
    s = np.asarray(s).reshape(-1)
    mu_cond = []
    cov_cond = []
    log_weights = []
    for k in range(num_components):
        mu = np.asarray(mu_k[k])
        sigma = np.asarray(sig_k[k]).copy()
        mu_s = mu[:dim_s]
        mu_xi = mu[dim_s:]
        sigma_ss = sigma[:dim_s, :dim_s].copy()
        sigma_sx = sigma[:dim_s, dim_s:].copy()
        sigma_xs = sigma[dim_s:, :dim_s].copy()
        sigma_xx = sigma[dim_s:, dim_s:].copy()
        sigma_ss_reg = sigma_ss + reg * np.eye(dim_s)
        try:
            sigma_ss_inv = np.linalg.inv(sigma_ss_reg)
        except np.linalg.LinAlgError:
            sigma_ss_inv = np.linalg.pinv(sigma_ss_reg)
        cond_mu = mu_xi + sigma_xs @ sigma_ss_inv @ (s - mu_s)
        cond_cov = sigma_xx - sigma_xs @ sigma_ss_inv @ sigma_sx
        cond_cov = 0.5 * (cond_cov + cond_cov.T)
        eigvals = np.linalg.eigvalsh(cond_cov)
        min_eig = eigvals.min()
        if min_eig < eig_floor:
            cond_cov += (eig_floor - min_eig + reg) * np.eye(cond_cov.shape[0])
        try:
            log_weight = (np.log(max(p_k[k], 1e-300))+multivariate_normal.logpdf(s, mean=mu_s, cov=sigma_ss_reg, allow_singular=True))
        except Exception:
            log_weight = -np.inf
        mu_cond.append(cond_mu)
        cov_cond.append(cond_cov)
        log_weights.append(log_weight)
    log_weights = np.asarray(log_weights)
    if not np.any(np.isfinite(log_weights)):
        weights = np.ones(num_components) / num_components
    else:
        max_log_weight = np.max(log_weights[np.isfinite(log_weights)])
        weights = np.exp(log_weights - max_log_weight)
        weights[~np.isfinite(weights)] = 0.0
        if weights.sum() <= 1e-12:
            weights = np.ones(num_components) / num_components
        else:
            weights = weights / weights.sum()

    return np.array(mu_cond), np.array(cov_cond), weights

def MC_sampling(K, N, mu_list, cov_list, p_list, seed=None):
    d = mu_list.shape[1]
    samples = np.zeros((N, d))
    rng = np.random.default_rng(seed)
    for i in range(N):
        k = rng.choice(K, p=p_list)
        samples[i] = rng.multivariate_normal(mu_list[k], cov_list[k])
    return samples

def oos_loss(q, s, h, b, W1, W2, dim_xi=1, seed=None):
    n = 100000
    rng = np.random.default_rng(seed)
    s = s.reshape(1, -1)
    eps1 = rng.uniform(-2, 2, size=(n, dim_xi))
    eps2 = rng.uniform(-2, 2, size=(n, dim_xi))
    lin  = s @ W1.T
    quad = (s**2) @ W2.T
    xi1 = lin + 50 + eps1
    xi2 = quad + 40 + eps2
    score = s[:, 0]
    prob = 0.5 * (1 + np.tanh(score))
    u = rng.random(n)
    xi_samples = np.where(u[:, None] < prob[:, None], xi1, xi2)
    q = np.array(q).reshape(1, -1)
    if q.shape[1] == 1 and dim_xi > 1:
        q = np.tile(q, (1, dim_xi))
    losses = h * np.maximum(q - xi_samples, 0) + b * np.maximum(xi_samples - q, 0)
    return np.mean(losses)

def oos_loss_valid(q, xi, h, b):
    loss = h * np.maximum(q - xi, 0) + b * np.maximum(xi - q, 0)
    return np.mean(loss)

In [ ]:
############################### RNW function #########################
def NW_weights(s_test, s_is, H):
    N = len(s_is)
    numerator = np.zeros(N)

    H = float(H)
    if not np.isfinite(H) or H <= 0:
        return np.ones(N) / N

    s_is_arr = np.asarray(s_is, dtype=float)
    if s_is_arr.ndim == 1:
        s_is_arr = s_is_arr.reshape(-1, 1)
    s_test_arr = np.asarray(s_test, dtype=float).reshape(-1)

    S = np.cov(s_is_arr.T, ddof=1)
    S = np.asarray(S, dtype=float)
    if S.ndim == 0:
        S = np.asarray([[float(S)]])
    S = S + 1e-8 * np.eye(S.shape[0])

    for i in range(N):
        diff = s_is_arr[i] - s_test_arr
        sol = np.linalg.solve(S, diff)
        quad = max(float(diff @ sol), 0.0)
        numerator[i] = np.exp(-0.5 * np.sqrt(quad / max(H, 1e-12)))

    denominator = numerator.sum()
    weight = numerator / denominator if denominator != 0 else np.ones(N) / N
    weight = weight / weight.sum() if weight.sum() != 0 else np.ones(N) / N
    return weight

In [ ]:
########################### ResDRO function ##############################
def NewsVendor_1_Wass(xi_is, eps, h, b):
    scale = 100
    xi_is = xi_is.astype(float) / scale
    eps = eps / scale

    N = xi_is.shape[0]

    lda = cp.Variable(nonneg=True)
    s = cp.Variable(N)
    z = cp.Variable((N, 2))
    q = cp.Variable(nonneg=True)

    const = []

    for i in range(N):
        const.append(h * q + z[i,0] * xi_is[i] <= s[i])
        const.append(-b * q + z[i,1] * xi_is[i] <= s[i])
        const.append(z[i,0] >= -h)
        const.append(z[i,1] >= b)
        for k in range(2):
            const.append(cp.norm_inf(z[i,k]) <= lda)

    obj = cp.Minimize(lda * eps + (1 / N) * cp.sum(s))
    prob = cp.Problem(obj, const)
    prob.solve(solver=cp.MOSEK)

    return q.value * scale

In [ ]:
############################### LDR function ###############################
def LDR(xi_is, s_is, h, b, verbose=False):
    xi_is = np.asarray(xi_is, dtype=float)
    s_is = np.asarray(s_is, dtype=float)
    N, dim_xi = xi_is.shape
    _, dim_s = s_is.shape

    beta0 = cp.Variable(dim_xi)
    B = cp.Variable((dim_s, dim_xi))

    xi_hat = s_is @ B + beta0

    h_vec = np.asarray(h, dtype=float)
    b_vec = np.asarray(b, dtype=float)

    excess = cp.Variable((N, dim_xi), nonneg=True)
    shortage = cp.Variable((N, dim_xi), nonneg=True)

    constraints = [excess >= xi_hat - xi_is, shortage >= xi_is - xi_hat]

    loss = cp.sum(cp.multiply(h_vec.reshape(1, -1), excess) + cp.multiply(b_vec.reshape(1, -1), shortage)) / N

    prob = cp.Problem(cp.Minimize(loss), constraints)
    prob.solve(solver=cp.MOSEK, verbose=verbose)

    if beta0.value is None or B.value is None:
        raise RuntimeError(f"LDR failed. Status: {prob.status}")

    rule = {
        "beta0": np.asarray(beta0.value).reshape(-1),
        "B": np.asarray(B.value),
        "objective": prob.value,
        "status": prob.status,
    }

    return rule

def LDR_decision(s_test, rule):
    s_test = np.asarray(s_test, dtype=float)
    beta0 = rule["beta0"]
    B = rule["B"]

    x = s_test @ B + beta0
    x = x.reshape(-1)

    x = np.maximum(x, 0)

    return x

In [ ]:
############################### NN-DR function ###############################
def decision_loss_fn(q_pred, xi_true, h_tensor, b_tensor):
    excess = torch.relu(q_pred - xi_true)
    shortage = torch.relu(xi_true - q_pred)

    loss = h_tensor * excess + b_tensor * shortage

    return loss.sum(dim=1).mean()

def NN(xi_is, s_is, h, b, hidden_node, hidden_layer, lr, weight_decay, total_epoch, batch_size, val_rate, patience, min_delta, 
       seed=None, device="cpu", verbose=False):
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    xi_is = np.asarray(xi_is, dtype=float)
    s_is = np.asarray(s_is, dtype=float)
    N, dim_xi = xi_is.shape
    N_s, dim_s = s_is.shape

    h_vec = np.asarray(h, dtype=float)
    b_vec = np.asarray(b, dtype=float)

    rng = np.random.default_rng(seed)
    indices = np.arange(N)
    rng.shuffle(indices)

    N_val = int(val_rate * N)
    N_val = max(1, min(N_val, N - 1))

    val_idx = indices[:N_val]
    train_idx = indices[N_val:]

    s_train = s_is[train_idx]
    xi_train = xi_is[train_idx]

    s_val = s_is[val_idx]
    xi_val = xi_is[val_idx]

    s_mean = np.zeros((1, dim_s))
    s_std = np.ones((1, dim_s))

    s_train_fit = s_train
    s_val_fit = s_val

    X_train = torch.tensor(s_train_fit, dtype=torch.float32).to(device)
    Y_train = torch.tensor(xi_train, dtype=torch.float32).to(device)

    X_val = torch.tensor(s_val_fit, dtype=torch.float32).to(device)
    Y_val = torch.tensor(xi_val, dtype=torch.float32).to(device)

    h_tensor = torch.tensor(h_vec.reshape(1, -1), dtype=torch.float32).to(device)
    b_tensor = torch.tensor(b_vec.reshape(1, -1), dtype=torch.float32).to(device)

    N_train = X_train.shape[0]

    if batch_size is None:
        batch_size = N_train

    layers = []
    input_dim = dim_s

    for _ in range(hidden_layer):
        layers.append(nn.Linear(input_dim, hidden_node))
        layers.append(nn.ReLU())
        input_dim = hidden_node

    layers.append(nn.Linear(input_dim, dim_xi))
    layers.append(nn.Softplus())

    model = nn.Sequential(*layers).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay,)

    best_val_loss = np.inf
    best_train_loss = np.inf
    best_state = None
    patience_counter = 0
    best_epoch = 0

    for epoch in range(total_epoch):
        model.train()

        perm = torch.randperm(N_train, device=device)
        train_loss_epoch = 0.0

        for start in range(0, N_train, batch_size):
            idx = perm[start:start + batch_size]

            x_batch = X_train[idx]
            y_batch = Y_train[idx]

            q_pred = model(x_batch)
            loss = decision_loss_fn(q_pred, y_batch, h_tensor, b_tensor)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_epoch += loss.item() * len(idx)

        train_loss_epoch /= N_train

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = decision_loss_fn(val_pred, Y_val, h_tensor, b_tensor).item()

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_train_loss = train_loss_epoch
            best_epoch = epoch + 1
            patience_counter = 0

            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    stopped_epoch = epoch + 1 if total_epoch > 0 else 0
    early_stopped = stopped_epoch < total_epoch

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()

    rule = {
        "model": model,
        "s_mean": s_mean,
        "s_std": s_std,
        "dim_s": dim_s,
        "dim_xi": dim_xi,
        "best_train_loss": best_train_loss,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "early_stopped": early_stopped,
        "device": device,
        "loss_type": "newsvendor_decision_loss",
    }

    return rule

def NN_decision(s_test, rule):
    s_test = np.asarray(s_test, dtype=float)

    model = rule["model"]
    s_mean = rule["s_mean"]
    s_std = rule["s_std"]
    device = rule["device"]

    s_fit = (s_test - s_mean) / s_std
    s_test_tensor = torch.tensor(s_fit, dtype=torch.float32,).to(device)
    model.eval()

    with torch.no_grad():
        x = model(s_test_tensor).detach().cpu().numpy()

    return x.reshape(-1)

In [8]:
# ----------------------- Sequential Runtime Experiment ----------------------------
T_RUNTIME = 20
DIM_LIST = [1, 5, 20]
N_LIST = [50, 100, 200, 400]
NUM_MC_LIST = [100]
dim_xi = 1
h, b = 10, 2

GMM_COVARIANCE_TYPE = "diag"
GMM_REG_COVAR = 1e-2
GMM_N_INIT = 10
USE_FLOW_FORWARD_FOR_SAMPLING = False

K_FIXED = 3
eps_NF_GMM = 0.0
eps_nonNF_GMM = 0.0
eps_ResDRO = 0.0
C_H_FIXED = 1.0
C_smart_FIXED = 0.0
lda_smart = 0.0

hidden_node = 32
hidden_layer = 1
block_size = 1
bins = 8
total_epoch = 500

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def tic():
    sync_cuda()
    return perf_counter()

def toc(t0):
    sync_cuda()
    return perf_counter() - t0

def gmm_covariances_as_full_matrices(gmm):
    if gmm.covariance_type == "diag":
        return np.array([np.diag(cov) for cov in gmm.covariances_])
    elif gmm.covariance_type == "full":
        return np.array(gmm.covariances_)
    else:
        raise ValueError(f"Unsupported covariance_type: {gmm.covariance_type}")

def make_result(
    dim_s, N, trial, num_mc, method, total_time_sec, fit_time_sec,
    sampling_decision_time_sec, success=True, error_message="",
    early_stopping_applicable=False, best_epoch=np.nan,
    stopped_epoch=np.nan, early_stopped=np.nan
):
    return {
        "dim_s": dim_s,
        "N": N,
        "trial": trial,
        "num_mc": num_mc,
        "method": method,
        "total_time_sec": total_time_sec,
        "fit_time_sec": fit_time_sec,
        "sampling_decision_time_sec": sampling_decision_time_sec,
        "early_stopping_applicable": early_stopping_applicable,
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "early_stopped": early_stopped,
        "success": success,
        "error_message": error_message,
    }

def run_one_setting(dim_s, N, trial):
    print(f"Running dim_s={dim_s}, N={N}, trial={trial}")

    seed = 1_000_000 + 10_000 * dim_s + 100 * N + trial
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    rng = np.random.default_rng(seed)
    seed_train = int(rng.integers(1_000_000_000))
    seed_test = int(rng.integers(1_000_000_000))
    seed_nn = int(rng.integers(1_000_000_000))

    W1 = 0.3 * np.ones((dim_xi, dim_s))
    W2 = 5 * np.ones((dim_xi, dim_s))

    s_is, xi_is = generate_data(N, dim_s, dim_xi, W1, W2, seed=seed_train)
    s_test, _ = generate_data(1, dim_s, dim_xi, W1, W2, seed=seed_test)
    s_test = s_test[0]

    results = []

    # ------------------------------- LDR -------------------------------
    method = "LDR"
    t0 = tic()
    try:
        ldr_rule = LDR(xi_is=xi_is, s_is=s_is, h=h, b=b, verbose=False)
        _ = LDR_decision(s_test=s_test, rule=ldr_rule)
        total_time = toc(t0)
        print(f"Finished {method}: {total_time:.2f} sec")
        results.append(make_result(dim_s, N, trial, np.nan, method, total_time, total_time, np.nan))
    except Exception as exc:
        total_time = toc(t0)
        print(f"Failed {method}: {total_time:.2f} sec - {exc}")
        results.append(make_result(dim_s, N, trial, np.nan, method, total_time, total_time, np.nan, False, str(exc)))

    # ------------------------------- NN-DR -------------------------------
    method = "NN-DR"
    t0 = tic()
    try:
        nn_rule = NN(xi_is=xi_is, s_is=s_is, h=h, b=b,
                     hidden_node=hidden_node, hidden_layer=hidden_layer, lr=1e-3, weight_decay=1e-2, total_epoch=total_epoch, batch_size=64,
                     val_rate=0.1, patience=10, min_delta=1e-4, seed=seed_nn, device=device, verbose=False,)
        _ = NN_decision(s_test=s_test, rule=nn_rule)
        total_time = toc(t0)
        print(f"Finished {method}: {total_time:.2f} sec")
        results.append(make_result(dim_s, N, trial, np.nan, method, total_time, total_time, np.nan))
    except Exception as exc:
        total_time = toc(t0)
        print(f"Failed {method}: {total_time:.2f} sec - {exc}")
        results.append(make_result(dim_s, N, trial, np.nan, method, total_time, total_time, np.nan, False, str(exc)))

    # -------------------------------- RNW --------------------------------
    method = "RNW"
    t0 = tic()
    try:
        C_H = C_H_FIXED
        H = C_H / (N ** (1 / 6))
        weight = NW_weights(s_test, s_is, H)

        RNW_model = Newsvendor(reg=lda_smart, verbose=False)
        RNW_model.fit({"b": b, "h": h, "s_test": s_test, "xi_is": xi_is, "weight": weight})
        _ = RNW_model.coef_
        total_time = toc(t0)
        print(f"Finished {method}: {total_time:.2f} sec")
        results.append(make_result(dim_s, N, trial, np.nan, method, total_time, total_time, np.nan))
    except Exception as exc:
        total_time = toc(t0)
        print(f"Failed {method}: {total_time:.2f} sec - {exc}")
        results.append(make_result(dim_s, N, trial, np.nan, method, total_time, total_time, np.nan, False, str(exc)))

    # --------------------------- ResDRO -----------------------------
    method = "ResDRO"
    t0 = tic()
    try:
        model = LinearRegression().fit(s_is, xi_is)
        residuals = xi_is - model.predict(s_is)
        f_hat = model.predict(np.atleast_2d(s_test)).item()
        xi_er = np.maximum(residuals + f_hat, 0)

        _ = NewsVendor_1_Wass(xi_er, eps_ResDRO, h=h, b=b)
        total_time = toc(t0)
        print(f"Finished {method}: {total_time:.2f} sec")
        results.append(make_result(dim_s, N, trial, np.nan, method, total_time, total_time, np.nan))
    except Exception as exc:
        total_time = toc(t0)
        print(f"Failed {method}: {total_time:.2f} sec - {exc}")
        results.append(make_result(dim_s, N, trial, np.nan, method, total_time, total_time, np.nan, False, str(exc)))

    # --------------------------- NF-GMM fit -------------------------------
    nf_fit = None
    method = "NF-GMM"
    t0 = tic()
    try:
        scaler_s_nf = StandardScaler()
        scaler_xi_nf = StandardScaler()
        s_std_nf = scaler_s_nf.fit_transform(s_is)
        xi_std_nf = scaler_xi_nf.fit_transform(xi_is)
        data_std_nf = np.hstack([s_std_nf, xi_std_nf])

        x_tensor = torch.tensor(data_std_nf, dtype=torch.float32, device=device)

        base_gmm_nf = GaussianMixture(
            n_components=K_FIXED,
            covariance_type=GMM_COVARIANCE_TYPE,
            n_init=GMM_N_INIT,
            reg_covar=GMM_REG_COVAR,
            random_state=seed,
        ).fit(data_std_nf)

        nfm, _, nf_training_info = train_nf_model(latent_size=dim_s + dim_xi, best_K=K_FIXED, hidden_node=hidden_node, hidden_layer=hidden_layer, 
                                num_bins=bins, block_size=block_size, total_epoch=total_epoch, x=x_tensor, device=device,
                                base_gmm=base_gmm_nf)

        s_vec_nf = scaler_s_nf.transform(s_test.reshape(1, -1)).ravel()

        nf_fit = {
            "scaler_s": scaler_s_nf,
            "scaler_xi": scaler_xi_nf,
            "nfm": nfm,
            "s_vec": s_vec_nf,
            "mu_base": base_gmm_nf.means_,
            "sig_base": gmm_covariances_as_full_matrices(base_gmm_nf),
            "p_base": base_gmm_nf.weights_,
            "base_gmm": base_gmm_nf,
            "training_info": nf_training_info,
        }

        nf_fit_time = toc(t0)
        print(f"Finished {method} fit: {nf_fit_time:.2f} sec")
    except Exception as exc:
        nf_fit_time = toc(t0)
        print(f"Failed {method} fit: {nf_fit_time:.2f} sec - {exc}")
        for num_mc in NUM_MC_LIST:
            results.append(make_result(dim_s, N, trial, num_mc, method, nf_fit_time, nf_fit_time, np.nan, False, str(exc)))

    if nf_fit is not None:
        for num_mc in NUM_MC_LIST:
            t0 = tic()
            try:
                mu_cond, cov_cond, w = transforming_conditional(
                    s=nf_fit["s_vec"],
                    num_components=K_FIXED,
                    mu_k=nf_fit["mu_base"],
                    sig_k=nf_fit["sig_base"],
                    p_k=nf_fit["p_base"],
                    dim_s=dim_s,
                )

                xi_sampled_std = MC_sampling(K_FIXED, num_mc, mu_cond, cov_cond, w, seed=seed + 10_000 + num_mc)
                if USE_FLOW_FORWARD_FOR_SAMPLING:
                    z_full = np.hstack([
                        np.repeat(nf_fit["s_vec"].reshape(1, -1), len(xi_sampled_std), axis=0),
                        xi_sampled_std
                    ])
                    z_tensor = torch.tensor(z_full, dtype=torch.float32, device=device)
                    x_gen_std = forward(nf_fit["nfm"], z_tensor)
                    xi_sampled_std = x_gen_std[:, dim_s:]

                xi_sampled = nf_fit["scaler_xi"].inverse_transform(xi_sampled_std)
                xi_sampled = np.maximum(xi_sampled, 0)
                if xi_sampled.shape != (num_mc, dim_xi):
                    raise ValueError(f"NF-GMM sampled xi shape {xi_sampled.shape}, expected {(num_mc, dim_xi)}")

                _ = NewsVendor_2_Wass(xi_sampled, eps_NF_GMM, h, b)
                sampling_decision_time = toc(t0)
                total_time = nf_fit_time + sampling_decision_time
                print(f"Finished {method} num_mc={num_mc} sampling/decision: {sampling_decision_time:.2f} sec")
                info = nf_fit.get("training_info", {})
                results.append(make_result(
                    dim_s, N, trial, num_mc, method, total_time, nf_fit_time,
                    sampling_decision_time,
                    early_stopping_applicable=True,
                    best_epoch=info.get("best_epoch", np.nan),
                    stopped_epoch=info.get("stopped_epoch", np.nan),
                    early_stopped=info.get("early_stopped", np.nan),
                ))
            except Exception as exc:
                sampling_decision_time = toc(t0)
                total_time = nf_fit_time + sampling_decision_time
                print(f"Failed {method} num_mc={num_mc} sampling/decision: {sampling_decision_time:.2f} sec - {exc}")
                results.append(make_result(dim_s, N, trial, num_mc, method, total_time, nf_fit_time, sampling_decision_time, False, str(exc)))

    # -------------------- nonNF-GMM fit --------------------------
    nonnf_fit = None
    method = "nonNF-GMM"
    t0 = tic()
    try:
        scaler_s_nonnf = StandardScaler()
        scaler_xi_nonnf = StandardScaler()
        s_std_nonnf = scaler_s_nonnf.fit_transform(s_is)
        xi_std_nonnf = scaler_xi_nonnf.fit_transform(xi_is)
        data_std_nonnf = np.hstack([s_std_nonnf, xi_std_nonnf])

        gmm_x = GaussianMixture(
            n_components=K_FIXED,
            covariance_type=GMM_COVARIANCE_TYPE,
            n_init=GMM_N_INIT,
            reg_covar=GMM_REG_COVAR,
            random_state=seed,
        ).fit(data_std_nonnf)

        s_vec = scaler_s_nonnf.transform(s_test.reshape(1, -1)).ravel()

        nonnf_fit = {
            "scaler_xi": scaler_xi_nonnf,
            "s_vec": s_vec,
            "mu_x": gmm_x.means_,
            "sig_x": gmm_covariances_as_full_matrices(gmm_x),
            "p_x": gmm_x.weights_,
        }

        nonnf_fit_time = toc(t0)
        print(f"Finished {method} fit: {nonnf_fit_time:.2f} sec")
    except Exception as exc:
        nonnf_fit_time = toc(t0)
        print(f"Failed {method} fit: {nonnf_fit_time:.2f} sec - {exc}")
        for num_mc in NUM_MC_LIST:
            results.append(make_result(dim_s, N, trial, num_mc, method, nonnf_fit_time, nonnf_fit_time, np.nan, False, str(exc)))

    if nonnf_fit is not None:
        for num_mc in NUM_MC_LIST:
            t0 = tic()
            try:
                mu_cond_x, cov_cond_x, p_cond_x = transforming_conditional(s=nonnf_fit["s_vec"], num_components=K_FIXED,
                                                                           mu_k=nonnf_fit["mu_x"], sig_k=nonnf_fit["sig_x"], p_k=nonnf_fit["p_x"], dim_s=dim_s)

                xi_sampled_std = MC_sampling(K_FIXED, num_mc, mu_cond_x, cov_cond_x, p_cond_x, seed=seed + 20_000 + num_mc)
                xi_sampled = nonnf_fit["scaler_xi"].inverse_transform(xi_sampled_std)
                xi_sampled = np.maximum(xi_sampled, 0)
                if xi_sampled.shape != (num_mc, dim_xi):
                    raise ValueError(f"nonNF-GMM sampled xi shape {xi_sampled.shape}, expected {(num_mc, dim_xi)}")

                _ = NewsVendor_2_Wass(xi_sampled, eps_nonNF_GMM, h, b)
                sampling_decision_time = toc(t0)
                total_time = nonnf_fit_time + sampling_decision_time
                print(f"Finished {method} num_mc={num_mc} sampling/decision: {sampling_decision_time:.2f} sec")
                results.append(make_result(dim_s, N, trial, num_mc, method, total_time, nonnf_fit_time, sampling_decision_time))
            except Exception as exc:
                sampling_decision_time = toc(t0)
                total_time = nonnf_fit_time + sampling_decision_time
                print(f"Failed {method} num_mc={num_mc} sampling/decision: {sampling_decision_time:.2f} sec - {exc}")
                results.append(make_result(dim_s, N, trial, num_mc, method, total_time, nonnf_fit_time, sampling_decision_time, False, str(exc)))

    return results

all_timing_results = []

for dim_s in DIM_LIST:
    for N in N_LIST:
        for trial in range(T_RUNTIME):
            all_timing_results.extend(run_one_setting(dim_s, N, trial))

out_dir = Path("Results")
out_dir.mkdir(parents=True, exist_ok=True)

raw_columns = [
    "dim_s",
    "N",
    "trial",
    "num_mc",
    "method",
    "total_time_sec",
    "fit_time_sec",
    "sampling_decision_time_sec",
    "early_stopping_applicable",
    "best_epoch",
    "stopped_epoch",
    "early_stopped",
    "success",
    "error_message",
]

raw_df = pd.DataFrame(all_timing_results, columns=raw_columns)

success_df = raw_df[raw_df["success"]].copy()

summary_df = (
    success_df
    .groupby(["dim_s", "N", "num_mc", "method"], dropna=False)
    .agg(
        avg_total_time_sec=("total_time_sec", "mean"),
        p10_total_time_sec=("total_time_sec", lambda x: np.percentile(x, 10)),
        p90_total_time_sec=("total_time_sec", lambda x: np.percentile(x, 90)),
        std_total_time_sec=("total_time_sec", "std"),
        avg_fit_time_sec=("fit_time_sec", "mean"),
        p10_fit_time_sec=("fit_time_sec", lambda x: np.percentile(x, 10)),
        p90_fit_time_sec=("fit_time_sec", lambda x: np.percentile(x, 90)),
        avg_sampling_decision_time_sec=("sampling_decision_time_sec", "mean"),
        p10_sampling_decision_time_sec=("sampling_decision_time_sec", lambda x: np.percentile(x, 10)),
        p90_sampling_decision_time_sec=("sampling_decision_time_sec", lambda x: np.percentile(x, 90)),
        avg_best_epoch=("best_epoch", "mean"),
        avg_stopped_epoch=("stopped_epoch", "mean"),
        early_stop_rate=("early_stopped", "mean"),
        n_success=("success", "count"),
    )
    .reset_index()
)

summary_path = out_dir / "NV_run_time.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved summarized timing results to:", summary_path)

gmm_summary_df = summary_df[summary_df["method"].isin(["NF-GMM", "nonNF-GMM"])].copy()
gmm_summary_path = out_dir / "NV_runtime_summary.csv"
gmm_summary_df.to_csv(gmm_summary_path, index=False)
print("Saved GMM num_mc timing summary to:", gmm_summary_path)

early_stop_df = (
    success_df[
        success_df["early_stopping_applicable"]
        & (success_df["method"] == "NF-GMM")
    ]
    .drop_duplicates(["dim_s", "N", "trial", "method"])
    .copy()
)

early_stop_summary_df = (
    early_stop_df
    .groupby(["dim_s", "N", "method"], dropna=False)
    .agg(
        avg_best_epoch=("best_epoch", "mean"),
        p10_best_epoch=("best_epoch", lambda x: np.percentile(x, 10)),
        p90_best_epoch=("best_epoch", lambda x: np.percentile(x, 90)),
        avg_stopped_epoch=("stopped_epoch", "mean"),
        p10_stopped_epoch=("stopped_epoch", lambda x: np.percentile(x, 10)),
        p90_stopped_epoch=("stopped_epoch", lambda x: np.percentile(x, 90)),
        early_stop_rate=("early_stopped", "mean"),
        n_success=("success", "count"),
    )
    .reset_index()
)

early_stop_summary_path = out_dir / "NV_early_stopping_summary.csv"
early_stop_summary_df.to_csv(early_stop_summary_path, index=False)
print("Saved early stopping summary to:", early_stop_summary_path)
display(early_stop_summary_df)


Finished NF-GMM fit: 0.98 sec
Finished NF-GMM num_mc=100 sampling/decision: 0.60 sec
Finished nonNF-GMM fit: 0.05 sec
Finished nonNF-GMM num_mc=100 sampling/decision: 0.62 sec
Saved summarized timing results to: Results/NV_run_time.csv
Saved GMM num_mc timing summary to: Results/NV_runtime_summary.csv
Saved early stopping summary to: Results/NV_early_stopping_summary.csv


,dim_s,N,method,avg_best_epoch,p10_best_epoch,p90_best_epoch,avg_stopped_epoch,p10_stopped_epoch,p90_stopped_epoch,early_stop_rate,n_success
0,1,50,NF-GMM,12.75,1.0,38.3,42.75,31.0,68.3,1.0,20
1,1,100,NF-GMM,30.65,1.0,101.2,60.65,31.0,131.2,1.0,20
2,1,200,NF-GMM,93.85,1.9,208.3,123.85,31.9,238.3,1.0,20
3,1,400,NF-GMM,97.90,34.8,163.8,127.90,64.8,193.8,1.0,20
4,5,50,NF-GMM,5.40,1.0,12.2,35.40,31.0,42.2,1.0,20
5,5,100,NF-GMM,15.50,1.0,33.3,45.50,31.0,63.3,1.0,20
6,5,200,NF-GMM,31.30,11.8,47.8,61.30,41.8,77.8,1.0,20
7,5,400,NF-GMM,39.55,18.9,67.3,69.55,48.9,97.3,1.0,20
8,20,50,NF-GMM,3.15,1.0,2.6,33.15,31.0,32.6,1.0,20
9,20,100,NF-GMM,11.65,1.0,23.2,41.65,31.0,53.2,1.0,20
